# Phase 2: Data Cleaning

This notebook cleans the raw transactions and accounts data by removing unnecessary rows (e.g., duplicates, missing critical IDs) and drops highly correlated/redundant columns if necessary, then saves the cleaned data to `data/cleaned/`.

In [ ]:
import pandas as pd
import numpy as np
import os

# Define paths
RAW_DATA_DIR = '../../data/original'
CLEANED_DATA_DIR = '../../data/cleaned'
INTERIM_DATA_DIR = '../../data/interim'

os.makedirs(CLEANED_DATA_DIR, exist_ok=True)
os.makedirs(INTERIM_DATA_DIR, exist_ok=True)

## 1. Load the Data

In [ ]:
transactions_path = os.path.join(RAW_DATA_DIR, 'transactions.csv')
accounts_path = os.path.join(RAW_DATA_DIR, 'accounts.csv')

tx_df = pd.read_csv(transactions_path)
acct_df = pd.read_csv(accounts_path)

print(f'Raw Transactions shape: {tx_df.shape}')
print(f'Raw Accounts shape: {acct_df.shape}')

## 2. Remove Duplicates

In [ ]:
tx_df_cleaned = tx_df.drop_duplicates()
acct_df_cleaned = acct_df.drop_duplicates()

print(f'Transactions shape after duplicate removal: {tx_df_cleaned.shape}')
print(f'Accounts shape after duplicate removal: {acct_df_cleaned.shape}')

## 3. Handle Missing Values / Unnecessary Rows

We will drop rows that are missing critical identifiers (e.g., if there is a transaction ID or Account ID column, it must be present). We will also check for completely empty rows.

In [ ]:
# Drop rows where all elements are NaN
tx_df_cleaned = tx_df_cleaned.dropna(how='all')
acct_df_cleaned = acct_df_cleaned.dropna(how='all')

# Identify ID columns dynamically or use known ones
tx_id_col = 'transaction_id' if 'transaction_id' in tx_df_cleaned.columns else tx_df_cleaned.columns[0]
acct_id_col = 'account_id' if 'account_id' in acct_df_cleaned.columns else acct_df_cleaned.columns[0]

# Drop rows missing the primary key
tx_df_cleaned = tx_df_cleaned.dropna(subset=[tx_id_col])
acct_df_cleaned = acct_df_cleaned.dropna(subset=[acct_id_col])

print(f'Transactions shape after missing ID removal: {tx_df_cleaned.shape}')
print(f'Accounts shape after missing ID removal: {acct_df_cleaned.shape}')

## 4. Remove Zero-Amount Transactions

For financial fraud detection, transactions with 0 or negative amount might be erroneous entries.

In [ ]:
amount_col = 'Amount' if 'Amount' in tx_df_cleaned.columns else ('amount' if 'amount' in tx_df_cleaned.columns else None)

if amount_col:
    tx_df_cleaned = tx_df_cleaned[tx_df_cleaned[amount_col] > 0]
    print(f'Transactions shape after dropping zero/negative amounts: {tx_df_cleaned.shape}')

## 5. Save Cleaned Data

We save the cleaned datasets into the `data/cleaned` directory as requested, and also to `data/interim` in parquet format as per AGENTS.md.

In [ ]:
tx_cleaned_csv = os.path.join(CLEANED_DATA_DIR, 'transactions.csv')
acct_cleaned_csv = os.path.join(CLEANED_DATA_DIR, 'accounts.csv')

tx_df_cleaned.to_csv(tx_cleaned_csv, index=False)
acct_df_cleaned.to_csv(acct_cleaned_csv, index=False)

tx_interim_pq = os.path.join(INTERIM_DATA_DIR, 'transactions.parquet')
acct_interim_pq = os.path.join(INTERIM_DATA_DIR, 'accounts.parquet')

# Save to parquet for efficiency in downstream tasks
tx_df_cleaned.to_parquet(tx_interim_pq, index=False)
acct_df_cleaned.to_parquet(acct_interim_pq, index=False)

print('Data cleaning complete. Files saved to:')
print(f'- {tx_cleaned_csv}')
print(f'- {tx_interim_pq}')